# Shift-Scan: what is left (`last.ipynb`)

Audit date: 2026-11-09, based on your Google Drive (`shift_scan_v2_results/`, `shift_scan_100m_runs/`,
the Colab notebooks, and the project summary).

## Already done (verified in Drive)
| Item | Evidence |
|---|---|
| v2 best config, 3 seeds @ 4M tokens | PPL 31.834 +/- 0.096 (`best_config.json`, `runs.jsonl`) |
| 12M-token run | PPL 29.283 |
| Per-layer projection sensitivity (48 projections) | `stage1_sensitivity.csv` |
| 1M-token training ablations | `stage2_ablation.csv` |
| Best checkpoint exists | `ckpt_v2_best_s0_t4000000.pt` (~673 MB) |
| Recovered 28-run archive | `shift_scan_runs_recovered (1).jsonl` |

## Still left (priority order)
| # | Gap | Why it matters |
|---|---|---|
| 1 | **Matched A0-A3 factorial with 3 seeds** | The PoT-decay claim has no evidence yet. Nearest/Dyadic2 differ from v2-best by ~0.03 PPL, far below the seed std of 0.096. Also v2-best used `decay: null`. |
| 2 | **Final_Experiments_Colab is an unexecuted scaffold** | `COMMAND_TEMPLATE` is empty, `RUN_EXPERIMENTS=False`, no outputs. |
| 3 | **100M-token HARPQ run never trained** | `log.jsonl` only contains step 0 (student PPL ~1.9e20). It is not evidence. |
| 4 | **Component sensitivity beyond projections** | Delta, B, C, decay, recurrent state not yet isolated. |
| 5 | **Long-context validation** | Nothing beyond training length yet. |
| 6 | **Zero-shot benchmarks** | LAMBADA only appears in old 10M runs (~2%). Need teacher, FP student, ternary. |
| 7 | **Real kernel speedup** | Triton kernel was ~82x slower than FP16 (12.98 ms vs 0.159 ms). |
| 8 | **Quality gap** | Ternary 31.83 vs FP student 23.10 at 4M tokens (+37.8%). |
| 9 | **Paper** | Draft with only claims the data supports. |

## 1. Setup and audit

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob, time, math, random, itertools
from pathlib import Path
import numpy as np, pandas as pd

ROOT   = Path('/content/drive/MyDrive/shift_scan_v2_results')
CKPT   = ROOT / 'ckpt_v2_best_s0_t4000000.pt'
BEST   = json.load(open(ROOT / 'best_config.json'))
OUT    = ROOT / 'last_phase'; OUT.mkdir(exist_ok=True)
RESULTS = OUT / 'runs_last.jsonl'          # append-only, resumable

print('checkpoint present:', CKPT.exists(), '|', round(CKPT.stat().st_size/1e6) if CKPT.exists() else 0, 'MB')
print('best config:', json.dumps(BEST, indent=1))

def append_jsonl(path, row):
    with open(path, 'a') as f:
        f.write(json.dumps(row) + '\n')

def read_jsonl(path):
    if not Path(path).exists(): return []
    return [json.loads(l) for l in open(path) if l.strip()]

In [ ]:
# Audit of the 100M run: does the log show any training beyond step 0?
log100 = Path('/content/drive/MyDrive/shift_scan_100m_runs/harpq_h128_ternary_100m_seed0/log.jsonl')
rows = read_jsonl(log100)
if rows:
    steps = [r.get('step', 0) for r in rows]
    print('rows:', len(rows), '| max step:', max(steps), '| last student_ppl:', rows[-1].get('student_ppl'))
    if max(steps) == 0:
        print('WARNING: the 100M run never advanced past step 0. Do not cite it as a result.')
else:
    print('log not found at', log100)

## 2. Wire your real training / eval code (do this once)

I could not see your v2 training function, so this cell is the only place you must edit.
Copy `train_and_eval` from `Shift_Scan_v2_Final_Colab.ipynb` and make it return a dict with
`ppl`, `ce`, `tokens_actual`, `runtime_s`, `peak_vram_gb`.

Rules for a fair comparison: identical data stream, validation set, tokenizer, init, optimizer,
schedule, token budget and seq length across every cell.

In [ ]:
TOKENS  = 4_000_000     # matches v2 best config, so results are comparable to 31.83
SEQ_LEN = 1024          # set this to whatever v2 used
SEEDS   = [0, 1, 2]

def train_and_eval(cfg: dict, seed: int) -> dict:
    """
    cfg keys: ternary ('twn' or None), decay (None | 'nearest' | 'harpq' | 'dyadic2'),
              plus everything in BEST['qcfg'] (ramp, learn_scale, fp_proj), lr, tokens.
    Must: set seed, build model, train for cfg['tokens'], evaluate on the SAME held-out
    FineWeb-Edu set as v2, and return a metrics dict.
    """
    raise NotImplementedError('Paste your v2 train/eval code here.')

## 3. Matched factorial: projections x decay (the main missing experiment)

Run the cheap, decisive comparisons first (`PRIORITY`). Everything is logged to an append-only JSONL, so a
Colab disconnect never loses finished runs.

In [ ]:
base_q = dict(BEST['qcfg'])

def make_cfg(ternary, decay):
    q = dict(base_q)
    q['ternary'] = ternary
    q['decay']   = decay
    if ternary is None:
        q['fp_proj'] = []          # nothing to protect when projections are FP
    return dict(qcfg=q, lr=BEST['lr'], tokens=TOKENS, seq_len=SEQ_LEN, model_size='130m')

CELLS = {
    'A0_fp_origdecay'      : make_cfg(None,  None),
    'A1_tern_origdecay'    : make_cfg('twn', None),
    'A2_fp_nearest'        : make_cfg(None,  'nearest'),
    'A2_fp_harpq'          : make_cfg(None,  'harpq'),
    'A2_fp_dyadic2'        : make_cfg(None,  'dyadic2'),
    'A3_tern_nearest'      : make_cfg('twn', 'nearest'),
    'A3_tern_harpq'        : make_cfg('twn', 'harpq'),
    'A3_tern_dyadic2'      : make_cfg('twn', 'dyadic2'),
}
# Most informative first: does PoT decay hurt FP? does it hurt ternary?
PRIORITY = ['A0_fp_origdecay', 'A1_tern_origdecay', 'A3_tern_harpq', 'A3_tern_nearest',
            'A2_fp_harpq', 'A2_fp_nearest', 'A3_tern_dyadic2', 'A2_fp_dyadic2']

RUN = False   # flip to True only after train_and_eval is wired
done = {(r['cell'], r['seed']) for r in read_jsonl(RESULTS) if r.get('status') == 'ok'}

if RUN:
    for name in PRIORITY:
        for seed in SEEDS:
            if (name, seed) in done: continue
            t0 = time.time()
            try:
                m = train_and_eval(CELLS[name], seed)
                row = dict(cell=name, seed=seed, tokens_budget=TOKENS, seq_len=SEQ_LEN,
                           status='ok', runtime_s=time.time()-t0, **m)
            except Exception as e:
                row = dict(cell=name, seed=seed, status='error', error=repr(e))
            append_jsonl(RESULTS, row)
            print(name, seed, row.get('ppl'), row['status'])
else:
    print('RUN=False. Planned runs:', len(CELLS) * len(SEEDS))

In [ ]:
# Aggregate + significance. Reports mean, std, and whether a gap exceeds seed noise.
from scipy import stats
df = pd.DataFrame([r for r in read_jsonl(RESULTS) if r.get('status') == 'ok'])
if len(df):
    summ = df.groupby('cell')['ppl'].agg(['count', 'mean', 'std']).round(4)
    display(summ)

    def compare(a, b):
        x, y = df[df.cell == a].sort_values('seed').ppl.values, df[df.cell == b].sort_values('seed').ppl.values
        if len(x) < 2 or len(y) < 2: return None
        t = stats.ttest_ind(x, y, equal_var=False)
        return dict(a=a, b=b, diff=float(y.mean() - x.mean()), p=float(t.pvalue))

    pairs = [('A1_tern_origdecay', 'A3_tern_harpq'), ('A1_tern_origdecay', 'A3_tern_nearest'),
             ('A1_tern_origdecay', 'A3_tern_dyadic2'), ('A3_tern_nearest', 'A3_tern_harpq'),
             ('A0_fp_origdecay', 'A2_fp_harpq'), ('A0_fp_origdecay', 'A2_fp_nearest'),
             ('A0_fp_origdecay', 'A1_tern_origdecay')]
    display(pd.DataFrame([c for c in (compare(a, b) for a, b in pairs) if c]))
    print('Only claim a decay effect if |diff| is clearly above the seed std AND p < 0.05 with 3+ seeds.')
else:
    print('No finished runs yet.')

## 4. Component sensitivity beyond projections (Phase C)

You already have per-layer projection sensitivity. Still missing: **Delta (dt), B, C, decay, state**.
The hooks below fake-quantize slices of `in_proj`'s output at N bits. Verify the split sizes against your
model: for Mamba-2 130M the output is `[z, x, B, C, dt]`.

In [ ]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def fake_quant(x, bits):
    qmax = 2 ** (bits - 1) - 1
    s = x.detach().abs().amax(dim=-1, keepdim=True).clamp_min(1e-8) / qmax
    return (x / s).round().clamp(-qmax, qmax) * s

def split_sizes(mixer):
    d_inner = getattr(mixer, 'd_inner', None) or mixer.intermediate_size
    d_state = getattr(mixer, 'd_state', None) or mixer.ssm_state_size
    ngroups = getattr(mixer, 'ngroups', None) or getattr(mixer, 'n_groups', 1)
    nheads  = getattr(mixer, 'nheads', None) or mixer.num_heads
    return dict(z=d_inner, x=d_inner, B=ngroups * d_state, C=ngroups * d_state, dt=nheads)

def attach_slice_quant(model, part, bits):
    handles = []
    for m in model.modules():
        if hasattr(m, 'in_proj') and (hasattr(m, 'd_state') or hasattr(m, 'ssm_state_size')):
            sz = split_sizes(m); names = list(sz); idx = np.cumsum([0] + [sz[n] for n in names])
            lo, hi = idx[names.index(part)], idx[names.index(part) + 1]
            def hook(mod, inp, out, lo=lo, hi=hi):
                out = out.clone(); out[..., lo:hi] = fake_quant(out[..., lo:hi], bits); return out
            handles.append(m.in_proj.register_forward_hook(hook))
    return handles

# Usage (needs your model + eval_ppl from section 5):
# for part in ['dt', 'B', 'C']:
#     for bits in [8, 6, 4]:
#         hs = attach_slice_quant(model, part, bits)
#         print(part, bits, eval_ppl(model, val_blocks))
#         [h.remove() for h in hs]
# Decay and recurrent-state quantization: reuse your existing nearest/harpq/dyadic2 quantizers and
# a state-rounding hook inside the scan. Log everything to RESULTS with cell='sens_<part>_<bits>'.

## 5. Long-context validation (Phase D)

Use the existing 4M checkpoint, so no retraining is needed. Evaluate A0 (FP student), A1 (ternary, original decay)
and any A3 variant at lengths beyond training length. Report loss by position and finite-value checks.
Keep one fixed validation stream and do not change training budget for this evaluation.

In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def eval_ppl(model, blocks, bs=4):
    model.eval(); tot, n = 0.0, 0
    for i in range(0, len(blocks), bs):
        x = torch.stack(blocks[i:i+bs]).to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            logits = model(x).logits
        loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction='sum')
        tot += loss.item(); n += x[:, 1:].numel()
    return math.exp(tot / n)

@torch.no_grad()
def loss_by_position(model, blocks, bucket=128, bs=1):
    model.eval(); per_pos = None; count = 0; nonfinite = 0
    for i in range(0, len(blocks), bs):
        x = torch.stack(blocks[i:i+bs]).to(dev)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
            logits = model(x).logits
        nonfinite += int((~torch.isfinite(logits)).sum().item())
        l = F.cross_entropy(logits[:, :-1].float().transpose(1, 2), x[:, 1:], reduction='none').mean(0)
        per_pos = l if per_pos is None else per_pos + l; count += 1
    per_pos = (per_pos / count).cpu().numpy()
    nb = len(per_pos) // bucket
    return dict(bucket_loss=[float(per_pos[j*bucket:(j+1)*bucket].mean()) for j in range(nb)], nonfinite=nonfinite)

def build_val_blocks(tokenizer, seq_len, n_blocks, val_modulo=100, max_docs=20000):
    # IMPORTANT: replace with your v2 validation builder so numbers stay comparable.
    from datasets import load_dataset
    ds = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT', split='train', streaming=True)
    buf, blocks = [], []
    for i, ex in enumerate(ds):
        if i >= max_docs: break
        if i % val_modulo != 0: continue
        buf += tokenizer(ex['text'])['input_ids'] + [tokenizer.eos_token_id]
        while len(buf) >= seq_len and len(blocks) < n_blocks:
            blocks.append(torch.tensor(buf[:seq_len])); buf = buf[seq_len:]
        if len(blocks) >= n_blocks: break
    return blocks

LENGTHS = [1024, 2048, 4096, 8192]
# for L in LENGTHS:
#     blocks = build_val_blocks(tok, L, n_blocks=16)
#     for name, model in {'A0': m_fp, 'A1': m_tern, 'A3': m_tern_pot}.items():
#         r = loss_by_position(model, blocks); append_jsonl(RESULTS, dict(cell='longctx_'+name, seq_len=L, status='ok', **r))

## 6. Zero-shot benchmarks

Compare teacher, FP student, and best ternary student on the same tasks. Needs `lm_eval` (0.4.13 is in your run manifest).

In [ ]:
# !pip -q install lm_eval==0.4.13
TASKS = ['lambada_openai', 'hellaswag', 'arc_easy', 'piqa', 'wikitext']

def run_zero_shot(model, tokenizer, tag):
    import lm_eval
    from lm_eval.models.huggingface import HFLM
    lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=8)
    res = lm_eval.simple_evaluate(model=lm, tasks=TASKS, num_fewshot=0)['results']
    append_jsonl(RESULTS, dict(cell='zeroshot_' + tag, status='ok', results=res))
    return res

## 7. Real hardware evaluation

Your Triton kernel was ~82x slower than FP16, so nothing here can claim a speedup yet. A fairer first step
is to test whether **ternary weights stored as int8 with int8 activations** (`torch._int_mm`, uses int8 tensor cores on a T4)
beats FP16. If that works, it is a deployable path today; a bit-packed shift/add kernel is a later optimization.

In [ ]:
def bench(fn, iters=50, warm=10):
    for _ in range(warm): fn()
    torch.cuda.synchronize()
    s, e = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

def kernel_bench(M, K, N):
    out = {}
    x16 = torch.randn(M, K, device='cuda', dtype=torch.float16)
    w16 = torch.randn(K, N, device='cuda', dtype=torch.float16)
    out['fp16_ms'] = bench(lambda: x16 @ w16)
    wt  = torch.randint(-1, 2, (K, N), device='cuda', dtype=torch.int8)
    x8  = torch.randint(-127, 128, (M, K), device='cuda', dtype=torch.int8)
    try:
        out['int8_ternary_ms'] = bench(lambda: torch._int_mm(x8, wt))
    except Exception as e:
        out['int8_ternary_ms'] = None; out['note'] = repr(e)[:120]
    return out

if torch.cuda.is_available():
    for (M, K, N) in [(512, 768, 3072), (2048, 768, 3072), (2048, 1536, 768)]:
        r = kernel_bench(M, K, N); r.update(M=M, K=K, N=N); print(r)
        append_jsonl(RESULTS, dict(cell='kernel_bench', status='ok', **r))
# Also measure end-to-end tokens/sec and peak VRAM for FP16 vs ternary model (model.generate or a scan-only loop).

## 8. Final consolidation and claim check

Fill this from `runs_last.jsonl` once the runs finish. Claim only what survives the check.

In [ ]:
rows = read_jsonl(RESULTS)
print('logged rows:', len(rows), '| ok:', sum(r.get('status') == 'ok' for r in rows))

CLAIMS = {
 'Ternary TWN + FP-sensitive layers is trainable (PPL ~31.8 @ 4M)'          : 'supported (3 seeds)',
 'PoT decay preserves quality (FP projections)'                              : 'needs A0 vs A2, 3 seeds',
 'PoT decay preserves quality (ternary projections)'                         : 'needs A1 vs A3, 3 seeds',
 'HARPQ beats Nearest PoT'                                                   : 'NOT supported (differences within noise)',
 'Long-context stability'                                                    : 'needs section 5',
 'Multiplication elimination (85% SSM / 62% whole model)'                    : 'analytical only; state as such',
 'Inference speedup'                                                         : 'NOT supported (kernel 82x slower)',
 'Novelty: ternary SSM'                                                      : 'NOT novel (prior work exists)',
 'Novelty: PoT decay + ternary + sensitivity + shift-add kernel, matched'    : 'only if A3 shows no loss vs A1',
}
display(pd.DataFrame(list(CLAIMS.items()), columns=['claim', 'status']))

## 9. Paper checklist
- [ ] Section 3 table: A0-A3, mean +/- std, 3 seeds, one token budget
- [ ] Per-layer sensitivity figure from `stage1_sensitivity.csv` (justifies the 6 FP-kept projections: `0:out, 13:in, 23:out, 0:in, 3:in, 23:in`)
- [ ] Training-recipe ablation table from `stage2_ablation.csv` (ramp and FP-sensitive are the big levers)
- [ ] Long-context loss-by-position plot
- [ ] Zero-shot table (teacher vs FP student vs ternary)
- [ ] Kernel/latency table with honest negative result if still slower
- [ ] Limitations: 130M scale, <=12M tokens, T4 only, lost checkpoints, 100M run not completed
- [ ] Do not claim HARPQ superiority unless Section 3 supports it